# BART-large MNLI — DIMER E2E zero-shot classification fine-tuning tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/bart-mnli-zero-shot-classification-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/bart-mnli-zero-shot-classification-pipeline/blob/main/tutorials/bart_zero_shot_classification_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-facebook%2Fbart--large--mnli-ffcc4d?style=flat)](https://huggingface.co/facebook/bart-large-mnli) [![Upstream](https://img.shields.io/badge/Upstream-facebookresearch%2Ffairseq-181717?style=flat&logo=github&logoColor=white)](https://github.com/facebookresearch/fairseq/tree/main/examples/bart) [![arXiv](https://img.shields.io/badge/arXiv-1910.13461-b31b1b.svg)](https://arxiv.org/abs/1910.13461)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** zero-shot text classification by NLI entailment (one text, a caller-supplied label set, one entailment-derived score per label, sorted; single-label softmax across labels or independent multi-label scores) and bounded supervised fine-tuning of the last decoder blocks and the NLI head on a labelled-text dataset, using the pinned BART-large MNLI weights

**This notebook is standalone.** It carries the repository's package (3 modules under `src/bart_zero_shot_classification_pipeline/`, at revision `e740a56647ca`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `d7645e127eaf1aefc7862fd59a17a5aa8558b8ce` (~1632 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned BART-large MNLI snapshot (safetensors, 1.6 GB), fetches the two digest-pinned Banking77 CSV files from the project repository (1.1 MB, no credential), keeps ten intents and draws 400 / 100 / 200 balanced training, validation and test messages from the release's own partition, classifies three synthetic sentences through the inference contract with an input manifest and a rejection probe, scores the frozen zero-shot model on the test messages with accuracy and macro-F1 beside the majority-class baseline, runs a bounded fine-tuning of the last two decoder blocks and the NLI head on entailment/contradiction pairs built from the training messages with validation-accuracy epoch selection, scores the held-out split again, classifies new messages with the adapted model, exports the adapter as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify label parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5). On CPU the whole path takes about six minutes of model time after the downloads; a CUDA runtime is used automatically when present.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to supply your own labelled texts as a CSV (columns `id`, `text`, `label`), a JSON array or a JSONL file of `{{id, text, label}}` records with 2..32 distinct labels written as short readable phrases. They pass through the same validation, seeded stratified text-disjoint split, baseline, fine-tuning, held-out evaluation, inference, artifact export and reload-parity cells as the Banking77 sample. The expected schema and the ceilings are stated in the Prerequisites and in Section 4, and uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

At inference each candidate label is written into a hypothesis (`This example is {{label}}.` by default), every (text, hypothesis) pair is encoded once as one BPE sequence, and the 407 M-parameter BART-large encoder-decoder with the three-way NLI head fine-tuned upstream on MultiNLI emits contradiction / neutral / entailment logits per pair; the carried module turns them into one score per label — a softmax over the entailment logits across the labels (`multi_label=False`, the default) or, per label, the entailment probability of its own [contradiction, entailment] pair (`multi_label=True`) — and the **default decision rule is `argmax`**. What the upstream checkpoint supplies is the NLI model and the tokenizer; what the carried pipeline module adds is manifest verification, input validation with named ceilings (a premise+hypothesis pair over `MAX_TEXT_TOKENS` is rejected, not truncated), the decision rule, a fixed output contract, and the `validate_inputs` and `evaluation_report` stage helpers. **The score is an entailment-derived softmax, not a calibrated probability**, and no threshold is shipped.

What this notebook adds to inference is **adaptation with gold labels**. The dataset is real and far from MultiNLI: Banking77 (Casanueva et al., 2020; CC BY 4.0), 13,083 customer-support messages over 77 fine-grained banking intents, of which the tutorial keeps **ten** and gives each a readable phrase (`card arrival`, `a lost or stolen card`, `the exchange rate`, …) so the hypothesis reads as English. Zero-shot NLI already does well on these ten (the build record measured 82.5 % accuracy on the test split), and the fine-tuning question is whether a bounded adaptation — every labelled message becomes one **entailment pair** with its gold phrase and one **contradiction pair** with a seeded wrong phrase, and only the last two decoder blocks and the NLI head train — closes the rest of the gap on held-out messages. Two metrics are implemented in the carried `metrics.py` (accuracy and **macro-F1** with per-label precision/recall) and the **majority-class baseline** shows where a classifier that does nothing sits. Nothing here is a quality claim about your labels: it is one seeded split of one corpus.

**Learning objectives:** install the pinned runtime; read what the carried pipeline, dataset and metrics modules guarantee; stage and digest-verify the immutable upstream snapshot; fetch a digest-pinned labelled corpus and validate and split it without leakage; classify through the public API with an explicit label set and hypothesis template and read the ranked scores correctly in single-label and multi-label mode; score the frozen zero-shot model against gold labels beside the majority baseline and read the per-label F1; run a bounded fine-tuning on entailment/contradiction pairs with explicit hyperparameters and validation-based epoch selection; evaluate on an independent test split; classify new messages; and export a safetensors adapter that reloads against the pinned base with verified parity.

**This notebook does not demonstrate:** a trained classification head over a fixed label vocabulary (the adapted model is still an NLI scorer over any label set), text generation or summarisation (the `bart-cnn-summarization-pipeline` sibling covers that), natural-language inference on caller-built premise/hypothesis pairs, sentence embeddings, token-level tagging, full-model or encoder fine-tuning, non-English text, any calibrated probability, and any claim that a Banking77 split stands in for your label set. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU (float32) and uses CUDA automatically when available. CPU is adequate: the build record measured 6 s to load and digest-verify the 1.6 GB snapshot, about 0.3 s per message for a ten-label classification (one minute for the 200-message test split) and about 65 s per training epoch over 800 NLI pairs plus a 100-message validation pass per epoch. The pinned `torch==2.14.0` install and the 1.6 GB checkpoint are the large downloads of the run.
- **Knowledge:** basic Python; what natural-language inference (entailment vs contradiction) is; why a softmax over entailment logits is a ranking signal and not a calibrated probability; what accuracy and macro-F1 measure and why a balanced test split makes the majority baseline equal to one over the number of labels.
- **Data contract:** records are `{{id, text, label}}` — a message and the readable phrase of its gold label — the text 1..8,000 characters and, paired with the longest hypothesis, at most 1,024 BPE tokens at inference, the label 1..100 characters, 2..32 distinct labels, ids matching `[A-Za-z0-9_.:-]{{1,64}}` and unique; a dataset needs 8..20,000 records; texts are de-duplicated case-insensitively before splitting so the same message never sits in two splits; during training only, pairs are truncated to 256 BPE tokens (inference never truncates — it rejects). BYOD accepts CSV, JSON or JSONL in that shape.
- **Validation is structural, not semantic:** nothing checks that a label phrase describes its messages or that the labels are mutually exclusive — a mislabelled corpus is fine-tuned on without complaint, and a label phrase the model cannot read as English will score badly zero-shot.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there — a customer-support log with its intent labels is exactly that. The default path uploads nothing.
- **External access (data):** besides the Hub, the default path fetches two pinned objects (`train.csv` 839,073 bytes, SHA-256 `b06e26ac…`; `test.csv` 239,961 bytes, SHA-256 `d12d6e3b…`) from `raw.githubusercontent.com` at the pinned `PolyAI-LDN/task-specific-datasets` commit over HTTPS, each refused on any mismatch before it is read; Banking77 is CC BY 4.0 (attribution: PolyAI; Casanueva et al., 2020).
- **External access:** the Hugging Face Hub only, to fetch the pinned `facebook/bart-large-mnli` snapshot (~1632 MB in total) at revision `d7645e127eaf…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'tokenizers==0.22.2',
    'huggingface-hub==0.36.2',
    'safetensors==0.8.0',
    'numpy==2.5.3',
]
NOTEBOOK_SOURCE = {
    'repository': 'bart-mnli-zero-shot-classification-pipeline',
    'repository_revision': 'e740a56647ca31c5866da4ae63acd81c0d5ebcb7',
    'embedded_module': 'src/bart_zero_shot_classification_pipeline/pipeline.py',
    'embedded_modules': ['src/bart_zero_shot_classification_pipeline/metrics.py', 'src/bart_zero_shot_classification_pipeline/pipeline.py', 'src/bart_zero_shot_classification_pipeline/samples.py'],
    'module_sha256': '737874544a2c83a170c71050019331d16da8383a814da59f4e59d10b05739ead',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/bart_zero_shot_classification_pipeline/` @ `e740a56647ca`)

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/bart_zero_shot_classification_pipeline/metrics.py`

In [ ]:
"""Classification metrics over a labelled dataset (accuracy, macro-F1, per-label precision/recall/F1) and the
majority-class baseline.

Metrics are exact-match comparisons of the predicted top label against the gold label string; macro-F1
averages the per-label F1 over the label vocabulary of the gold labels (a label never predicted scores 0).
Both are reported in percent. Neither is a calibration measure: the score behind a top label is an
entailment-derived softmax, not a probability, and nothing here calibrates it.
"""

from __future__ import annotations

from collections import Counter
from collections.abc import Mapping, Sequence
from typing import Any

METRIC_DEFINITIONS = {
    "accuracy": (
        "fraction of items whose predicted top label equals the gold label (exact string match); percent"
    ),
    "macro_f1": (
        "unweighted mean over the gold label vocabulary of the per-label F1 (precision and recall of "
        "predicting that label); a label never predicted contributes 0; percent"
    ),
}


def classification_metrics(predicted: Sequence[str], gold: Sequence[str]) -> dict[str, Any]:
    """Accuracy, macro-F1 and per-label counts over parallel predicted and gold labels."""
    if len(predicted) != len(gold):
        raise ValueError(f"{len(predicted)} predictions but {len(gold)} gold labels")
    if not gold:
        raise ValueError("no items to score")
    labels = sorted(set(gold))
    per_label = {}
    for label in labels:
        tp = sum(1 for p, g in zip(predicted, gold, strict=True) if p == label and g == label)
        fp = sum(1 for p, g in zip(predicted, gold, strict=True) if p == label and g != label)
        fn = sum(1 for p, g in zip(predicted, gold, strict=True) if p != label and g == label)
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        per_label[label] = {
            "support": tp + fn,
            "predicted": tp + fp,
            "precision": 100.0 * precision,
            "recall": 100.0 * recall,
            "f1": 100.0 * f1,
        }
    correct = sum(1 for p, g in zip(predicted, gold, strict=True) if p == g)
    return {
        "n": len(gold),
        "accuracy": 100.0 * correct / len(gold),
        "macro_f1": sum(v["f1"] for v in per_label.values()) / len(per_label),
        "n_labels": len(labels),
        "per_label": per_label,
        "predicted_outside_gold_vocabulary": sum(1 for p in predicted if p not in per_label),
        "definitions": dict(METRIC_DEFINITIONS),
    }


def majority_baseline(records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """Predict the most frequent gold label for every item (ties broken alphabetically)."""
    gold = [str(r["label"]) for r in records]
    counts = Counter(gold)
    top = sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))[0][0]
    result = classification_metrics([top] * len(gold), gold)
    result["baseline"] = f"majority class ({top!r} for every item)"
    return result

**Module 2/3:** `src/bart_zero_shot_classification_pipeline/pipeline.py` (carried verbatim; see the note above)

In [ ]:
"""Zero-shot text classification by NLI entailment over the pinned ``facebook/bart-large-mnli``.

Weights load only from a digest-verified local snapshot (``weights/<key>/``) or, when explicitly allowed,
from the Hugging Face Hub at the pinned revision. One task method, ``classify``: the text is the NLI
premise, each caller-supplied label is turned into a hypothesis with ``HYPOTHESIS_TEMPLATE``, and the
entailment/contradiction logits are converted to one score per label (Yin et al., arXiv:1909.00161).

The adaptation contract (``evaluate``, ``adapt``, ``save_artifact``, ``from_artifact``) fine-tunes the last
decoder blocks and the NLI classification head on a validated ``{id, text, label}`` dataset — every labelled
text becomes one entailment pair (its gold label's hypothesis) and one contradiction pair (a seeded wrong
label's hypothesis) — with validation-accuracy epoch selection, and exports the trained tensors as a
safetensors adapter bound to the pinned base weights. The inference contract above is unchanged by it.
"""

from __future__ import annotations

import hashlib
import json
import math
import random
import time
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import numpy as np

MODEL_ID = "facebook/bart-large-mnli"
MODEL_REVISION = "d7645e127eaf1aefc7862fd59a17a5aa8558b8ce"
MODEL_LICENSE = "mit"
MODEL_KEY = "bart-large-mnli"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# Upstream hypothesis template (snapshot README "With manual PyTorch"): ``f'This example is {label}.'``.
HYPOTHESIS_TEMPLATE = "This example is {}."
# NLI head layout from the snapshot config.json ``label2id``: contradiction 0, neutral 1, entailment 2.
CONTRADICTION_INDEX = 0
ENTAILMENT_INDEX = 2
NUM_NLI_LABELS = 3
# Ceilings. 1024 is max_position_embeddings in config.json and model_max_length in tokenizer_config.json;
# a premise+hypothesis pair past it is rejected (not truncated) so a label is never scored on a cut premise.
MAX_TEXT_TOKENS = 1024
MAX_TEXT_CHARS = 8_000  # pre-tokenisation guard on the premise; ~4 chars per BPE token on English text
MAX_LABELS = 32  # hypotheses scored per classify() call (one forward pass, batched)
MAX_LABEL_CHARS = 100
DECISION_RULE_SINGLE = (
    "multi_label=False: softmax over the entailment logit across labels, argmax picks the label; "
    "no minimum score"
)
DECISION_RULE_MULTI = (
    "multi_label=True (or a single label): per label, softmax over [contradiction, entailment] logits, "
    "entailment probability is the score; no threshold applied"
)
WEIGHT_FILE = "model.safetensors"
WEIGHT_SHA256 = (
    "cfbb687dbbd9df99fe865e1860350a22aebac4d26ee4bcb50217f1df606a018e"  # manifest digest of WEIGHT_FILE
)
PARAMETER_COUNT = 407_344_131
DECODER_LAYERS = 12  # config.json decoder_layers
DEFAULT_TRAINABLE_DECODER_LAYERS = 2  # the last two decoder blocks plus the NLI head (34,646,019 parameters)
MAX_TRAIN_PAIR_TOKENS = 256  # premise+hypothesis truncation ceiling during adaptation (never at inference)
MAX_EVAL_RECORDS = 2_000
MIN_SCORED_RECORDS = 50  # below this a scored set is labelled a small sample
ARTIFACT_FORMAT = "org.valcorza.bart-large-mnli.adapter.v1"
ARTIFACT_FORMAT_VERSION = "1.0"
ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"
ARTIFACT_MANIFEST_NAME = "manifest.json"


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _read_manifest(root: Path) -> dict[str, Any]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        return json.load(fh)


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest.get("files", []):
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {"path": str(root), **manifest}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def _check_text(text: Any, name: str = "text") -> str:
    if not isinstance(text, str):
        raise TypeError(f"{name} must be str, got {type(text).__name__}")
    if not text.strip():
        raise ValueError(f"{name} is empty")
    if len(text) > MAX_TEXT_CHARS:
        raise ValueError(f"{name} has {len(text)} chars; ceiling is MAX_TEXT_CHARS={MAX_TEXT_CHARS}")
    return text


def _check_labels(labels: Any) -> list[str]:
    """``classify``'s label contract; raise naming the first violated ceiling."""
    if isinstance(labels, str | bytes) or not isinstance(labels, Sequence):
        raise TypeError("labels must be a list of str, not a single string")
    if not 1 <= len(labels) <= MAX_LABELS:
        raise ValueError(f"labels must hold 1..MAX_LABELS={MAX_LABELS} items, got {len(labels)}")
    clean = []
    for i, label in enumerate(labels):
        if not isinstance(label, str):
            raise TypeError(f"labels[{i}] must be str, got {type(label).__name__}")
        if not label.strip():
            raise ValueError(f"labels[{i}] is empty")
        if len(label) > MAX_LABEL_CHARS:
            raise ValueError(f"labels[{i}] has {len(label)} chars; ceiling MAX_LABEL_CHARS={MAX_LABEL_CHARS}")
        clean.append(label)
    if len(set(clean)) != len(clean):
        raise ValueError("labels must be unique")
    return clean


def _check_template(template: Any) -> str:
    if not isinstance(template, str):
        raise TypeError("hypothesis_template must be str")
    if template.count("{}") != 1:
        raise ValueError("hypothesis_template must contain exactly one '{}' placeholder")
    return template


def _check_input_tokens(n_tokens: Sequence[int]) -> int:
    """The pair-token ceiling, applied once the tokenizer has counted every premise+hypothesis pair."""
    longest = max(int(n) for n in n_tokens)
    if longest > MAX_TEXT_TOKENS:
        raise ValueError(
            f"a premise+hypothesis pair is {longest} tokens; ceiling is MAX_TEXT_TOKENS={MAX_TEXT_TOKENS}"
        )
    return longest


INPUT_SCHEMA: dict[str, Any] = {
    "input": "one non-empty str (the NLI premise) and a list of unique non-empty str labels",
    "text_chars": [1, MAX_TEXT_CHARS],
    "pair_tokens": [1, MAX_TEXT_TOKENS],
    "labels": [1, MAX_LABELS],
    "label_chars": [1, MAX_LABEL_CHARS],
    "hypothesis_template": HYPOTHESIS_TEMPLATE,
    "multi_label": [False, True],
    "decision_rule": {"single": DECISION_RULE_SINGLE, "multi": DECISION_RULE_MULTI},
    "preprocessing": (
        "each label is inserted into hypothesis_template; every (premise, hypothesis) pair is BPE-encoded "
        "as one sequence with no truncation — a pair over MAX_TEXT_TOKENS is rejected, never cut"
    ),
}


def validate_inputs(
    texts: Sequence[str],
    labels: Sequence[str],
    *,
    multi_label: bool = False,
    hypothesis_template: str = HYPOTHESIS_TEMPLATE,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, per-input observations, verdict).

    ``classify`` takes one text per call, so ``texts`` is the batch the notebook will loop over; every
    entry and the shared ``labels``/``hypothesis_template`` go through the same private checks the
    method uses (``_check_text``, ``_check_labels``, ``_check_template``), so a rejection here is a
    rejection there. ``MAX_TEXT_TOKENS`` needs the loaded tokenizer and is enforced inside ``classify``.
    """
    if isinstance(texts, str | bytes) or not isinstance(texts, Sequence):
        raise TypeError("texts must be a sequence of str, not a single string")
    if not texts:
        raise ValueError("texts must hold at least one item")
    checked = [_check_text(text, f"texts[{i}]") for i, text in enumerate(texts)]
    clean_labels = _check_labels(labels)
    template = _check_template(hypothesis_template)
    if not isinstance(multi_label, bool):
        raise TypeError("multi_label must be a bool")
    if names is not None and len(names) != len(checked):
        raise ValueError("names must have one entry per text")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {"id": names[i] if names else f"text-{i}", "chars": len(text)} for i, text in enumerate(checked)
        ],
        "labels": clean_labels,
        "hypotheses": [template.format(label) for label in clean_labels],
        "multi_label": multi_label,
        "hypothesis_template": template,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def accuracy(predicted: Sequence[str], gold: Sequence[str]) -> float:
    """Fraction of items whose top label equals the gold label (exact string match)."""
    if len(predicted) != len(gold):
        raise ValueError("predicted and gold must have the same length")
    if not gold:
        raise ValueError("accuracy needs at least one item")
    return float(sum(p == g for p, g in zip(predicted, gold, strict=True)) / len(gold))


def evaluation_report(
    results: Sequence[Mapping[str, Any]],
    gold_labels: Sequence[str] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: ``accuracy`` of the top label against caller-supplied gold labels, else
    ``not-measurable``. The number is a sample-sanity observation on however many items were passed,
    never a benchmark; the score behind it is entailment-derived and not calibrated."""
    predicted = [str(result.get("top_label")) for result in results]
    supplied = gold_labels is not None
    report: dict[str, Any] = {
        "task": "zero-shot text classification by NLI entailment (caller-supplied label set)",
        "score_semantics": (
            "score is an entailment-derived softmax — over labels when multi_label=False, over "
            "[contradiction, entailment] per label otherwise — a ranking signal, not a calibrated "
            "probability; the decision rule is argmax over score and no threshold is shipped"
        ),
        "sample_kind": sample_kind,
        "n_items": len(predicted),
        "metrics": [],
        "baselines": [],
        "verdict": "not-measurable",
        "reason": "no gold labels were supplied, so the top labels cannot be scored",
        "needs": (
            "one gold label per text from the deployment's own label set over enough texts to state a "
            "dispersion; the `accuracy` helper then scores exact top-label matches, and a calibration set "
            "is needed before any score is read as a probability"
        ),
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if supplied:
        value = accuracy(predicted, list(gold_labels))
        report["metrics"] = [
            {
                "id": "accuracy",
                "value": value,
                "estimation": f"single pass over {len(predicted)} item(s); no dispersion",
                "decision_rule": "top_label == gold label (exact string match)",
            }
        ]
        report["verdict"] = "sample-sanity"
        report["reason"] = (
            f"gold labels were supplied for {len(predicted)} item(s); the accuracy is a plumbing check on "
            "that sample, not a benchmark"
        )
    return report


@dataclass
class BARTZeroShotClassificationPipeline:
    """``_runner(text, hypotheses)`` -> (NLI logits ``(n_hypotheses, 3)``, per-pair token counts). Injectable
    so tests run offline."""

    _runner: Callable[[str, list[str]], tuple[np.ndarray, list[int]]]
    device: str = "cpu"
    source: str = "injected"
    adapter: dict[str, Any] | None = field(default=None, repr=False)
    _model: Any = field(default=None, repr=False)
    _tokenizer: Any = field(default=None, repr=False)

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> BARTZeroShotClassificationPipeline:
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            location, kwargs, source = str(root), {"local_files_only": True}, "local-snapshot"
        elif allow_download:
            location, kwargs, source = MODEL_ID, {}, "hf-hub"
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage {MODEL_ID}@{MODEL_REVISION} under weights/{MODEL_KEY}"
            )
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import AutoTokenizer, BartForSequenceClassification

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        tokenizer = AutoTokenizer.from_pretrained(
            location, revision=MODEL_REVISION, trust_remote_code=False, **kwargs
        )
        model = BartForSequenceClassification.from_pretrained(
            location, revision=MODEL_REVISION, dtype=torch.float32, trust_remote_code=False, **kwargs
        )
        model = model.to(resolved_device).eval()

        def runner(text: str, hypotheses: list[str]) -> tuple[np.ndarray, list[int]]:
            batch = tokenizer(
                [text] * len(hypotheses), hypotheses, return_tensors="pt", padding=True, truncation=False
            )
            n_tokens = [int(n) for n in batch["attention_mask"].sum(dim=1)]
            _check_input_tokens(n_tokens)
            with torch.inference_mode():
                logits = model(**batch.to(resolved_device)).logits
            return logits.float().cpu().numpy(), n_tokens

        return cls(runner, resolved_device, source, _model=model, _tokenizer=tokenizer)

    def classify(
        self,
        text: str,
        labels: Sequence[str],
        *,
        multi_label: bool = False,
        hypothesis_template: str = HYPOTHESIS_TEMPLATE,
    ) -> dict[str, Any]:
        """Score every label against ``text``; ``labels`` in the result are sorted by descending score."""
        text = _check_text(text)
        clean = _check_labels(labels)
        template = _check_template(hypothesis_template)
        if not isinstance(multi_label, bool):
            raise TypeError("multi_label must be a bool")
        hypotheses = [template.format(label) for label in clean]
        logits, n_tokens = self._runner(text, hypotheses)
        logits = np.asarray(logits, dtype=np.float64)
        if logits.shape != (len(clean), NUM_NLI_LABELS):
            raise RuntimeError(f"backend returned {logits.shape}, expected ({len(clean)}, {NUM_NLI_LABELS})")
        longest = _check_input_tokens(n_tokens)
        # Upstream ZeroShotClassificationPipeline rule: a single label always takes the per-label path.
        per_label = multi_label or len(clean) == 1
        if per_label:
            pair = logits[:, [CONTRADICTION_INDEX, ENTAILMENT_INDEX]]
            shifted = np.exp(pair - pair.max(axis=1, keepdims=True))
            scores = (shifted / shifted.sum(axis=1, keepdims=True))[:, 1]
        else:
            entail = logits[:, ENTAILMENT_INDEX]
            shifted = np.exp(entail - entail.max())
            scores = shifted / shifted.sum()
        order = np.argsort(-scores, kind="stable")
        ranked = [
            {
                "label": clean[i],
                "score": float(scores[i]),
                "entailment_logit": float(logits[i, ENTAILMENT_INDEX]),
                "contradiction_logit": float(logits[i, CONTRADICTION_INDEX]),
            }
            for i in order
        ]
        return {
            "labels": ranked,
            "top_label": ranked[0]["label"],
            "multi_label": multi_label,
            "hypothesis_template": template,
            "decision_rule": DECISION_RULE_MULTI if per_label else DECISION_RULE_SINGLE,
            "n_tokens": longest,
            "device": self.device,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    # ---- adaptation -----------------------------------------------------------------------------------

    def _require_model(self) -> tuple[Any, Any]:
        if self._model is None or self._tokenizer is None:
            raise ValueError(
                "this operation needs a pipeline built with from_pretrained() or from_artifact()"
            )
        return self._model, self._tokenizer

    def evaluate(
        self,
        records: Sequence[Mapping[str, Any]],
        labels: Sequence[str] | None = None,
        *,
        multi_label: bool = False,
        hypothesis_template: str = HYPOTHESIS_TEMPLATE,
    ) -> dict[str, Any]:
        """Classify every record's text over `labels` (default: the dataset's own label vocabulary) and
        score the top labels against the gold labels (accuracy, macro-F1)."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import classification_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import label_names, validate_dataset` removed — names are kernel globals defined by the carried modules

        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS, labels=labels)[
            "records"
        ]
        label_list = list(labels) if labels is not None else label_names(checked)
        started = time.perf_counter()
        predicted = [
            self.classify(
                r["text"], label_list, multi_label=multi_label, hypothesis_template=hypothesis_template
            )["top_label"]
            for r in checked
        ]
        metrics = classification_metrics(predicted, [r["label"] for r in checked])
        metrics.update(
            {
                "labels": label_list,
                "multi_label": multi_label,
                "hypothesis_template": hypothesis_template,
                "verdict": "measured" if len(checked) >= MIN_SCORED_RECORDS else "measured-small-sample",
                "adapted": self.adapter is not None,
                "seconds": round(time.perf_counter() - started, 3),
                "model_id": MODEL_ID,
                "model_revision": MODEL_REVISION,
            }
        )
        return metrics

    def _trainable_names(self, trainable_decoder_layers: int) -> list[str]:
        if (
            not isinstance(trainable_decoder_layers, int)
            or not 1 <= trainable_decoder_layers <= DECODER_LAYERS
        ):
            raise ValueError(f"trainable_decoder_layers must be an int in 1..{DECODER_LAYERS}")
        model, _ = self._require_model()
        first = DECODER_LAYERS - trainable_decoder_layers
        prefixes = tuple(f"model.decoder.layers.{k}." for k in range(first, DECODER_LAYERS)) + (
            "classification_head.",
        )
        return [name for name, _p in model.named_parameters() if name.startswith(prefixes)]

    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None = None,
        *,
        labels: Sequence[str] | None = None,
        epochs: int = 2,
        lr: float = 2e-5,
        batch_size: int = 16,
        trainable_decoder_layers: int = DEFAULT_TRAINABLE_DECODER_LAYERS,
        seed: int = 0,
        hypothesis_template: str = HYPOTHESIS_TEMPLATE,
        progress: Callable[[dict[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded supervised fine-tuning of the NLI classifier on a validated labelled-text dataset.

        Every training record becomes two NLI pairs: (text, template(gold label)) labelled entailment and
        (text, template(a seeded wrong label from `labels`)) labelled contradiction. Only the last
        `trainable_decoder_layers` decoder blocks and the classification head train (2 blocks by default;
        the encoder, the embeddings and the earlier decoder blocks stay frozen). Cross-entropy over the three
        NLI logits, AdamW at a fixed learning rate with gradient clipping at 1.0, pairs truncated to
        MAX_TRAIN_PAIR_TOKENS **during training only**. Epoch 0 records the frozen model's validation
        accuracy; every epoch is scored on the validation split with `evaluate` under the same labels and
        template, and the epoch with the highest validation accuracy is kept."""
        pass  # standalone rewrite (build_notebook.py): `from .samples import label_names, validate_dataset` removed — names are kernel globals defined by the carried modules

        if not isinstance(epochs, int) or not 1 <= epochs <= 20:
            raise ValueError("epochs must be an int in 1..20")
        if not (0.0 < lr <= 1e-3):
            raise ValueError("lr must be in (0, 1e-3]")
        if not isinstance(batch_size, int) or not 1 <= batch_size <= 64:
            raise ValueError("batch_size must be an int in 1..64")
        template = _check_template(hypothesis_template)
        names = self._trainable_names(trainable_decoder_layers)
        train_checked = validate_dataset(train, labels=labels)["records"]
        label_list = _check_labels(list(labels) if labels is not None else label_names(train_checked))
        val_checked = (
            validate_dataset(val, min_records=1, max_records=MAX_EVAL_RECORDS, labels=label_list)["records"]
            if val
            else []
        )
        import torch

        torch.manual_seed(seed)
        rng = random.Random(seed)
        model, tokenizer = self._require_model()
        started = time.perf_counter()
        wanted = set(names)
        for name, param in model.named_parameters():
            param.requires_grad_(name in wanted)
        params = [p for p in model.parameters() if p.requires_grad]
        n_trainable = sum(p.numel() for p in params)
        optimiser = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
        device = torch.device(self.device)

        def score_val() -> dict[str, Any] | None:
            if not val_checked:
                return None
            model.eval()
            return {
                k: v
                for k, v in self.evaluate(val_checked, label_list, hypothesis_template=template).items()
                if k in ("accuracy", "macro_f1", "n")
            }

        history: list[dict[str, Any]] = []
        entry: dict[str, Any] = {"epoch": 0, "train_loss": None, "val": score_val(), "note": "frozen model"}
        history.append(entry)
        if progress:
            progress(entry)
        best_acc = entry["val"]["accuracy"] if entry["val"] else -math.inf
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
        best_epoch = 0
        generator = torch.Generator().manual_seed(seed)
        for epoch in range(1, epochs + 1):
            model.train()
            pairs: list[tuple[str, str, int]] = []
            for record in train_checked:
                wrong = rng.choice([lab for lab in label_list if lab != record["label"]])
                pairs.append((record["text"], template.format(record["label"]), ENTAILMENT_INDEX))
                pairs.append((record["text"], template.format(wrong), CONTRADICTION_INDEX))
            order = torch.randperm(len(pairs), generator=generator).tolist()
            losses = []
            for start in range(0, len(order), batch_size):
                batch = [pairs[i] for i in order[start : start + batch_size]]
                encoded = tokenizer(
                    [b[0] for b in batch],
                    [b[1] for b in batch],
                    return_tensors="pt",
                    padding=True,
                    truncation=True,
                    max_length=MAX_TRAIN_PAIR_TOKENS,
                )
                out = model(
                    input_ids=encoded["input_ids"].to(device),
                    attention_mask=encoded["attention_mask"].to(device),
                    labels=torch.tensor([b[2] for b in batch], device=device),
                )
                optimiser.zero_grad(set_to_none=True)
                out.loss.backward()
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                optimiser.step()
                losses.append(float(out.loss.detach()))
            model.eval()
            entry = {"epoch": epoch, "train_loss": sum(losses) / len(losses), "val": score_val()}
            history.append(entry)
            if progress:
                progress(entry)
            current = entry["val"]["accuracy"] if entry["val"] else math.inf
            if current > best_acc or not entry["val"]:
                best_acc = current
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
                best_epoch = epoch
        merged = dict(model.state_dict())
        merged.update(best_state)
        model.load_state_dict(merged, strict=True)
        model.eval()
        for param in model.parameters():
            param.requires_grad_(False)
        self.adapter = {
            "trainable_decoder_layers": trainable_decoder_layers,
            "trainable_names": names,
            "n_trainable": n_trainable,
            "n_total": sum(p.numel() for p in model.parameters()),
            "epochs": epochs,
            "best_epoch": best_epoch,
            "selection": "highest validation accuracy"
            if val_checked
            else "final epoch (no validation split)",
            "lr": lr,
            "batch_size": batch_size,
            "max_train_pair_tokens": MAX_TRAIN_PAIR_TOKENS,
            "labels": label_list,
            "hypothesis_template": template,
            "pairs_per_record": 2,
            "n_train": len(train_checked),
            "n_val": len(val_checked),
            "seed": seed,
            "history": history,
            "seconds": round(time.perf_counter() - started, 2),
        }
        return dict(self.adapter)

    # ---- artifacts ------------------------------------------------------------------------------------

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Write the adapted decoder-block and head tensors as safetensors with a manifest naming the base."""
        if self.adapter is None:
            raise ValueError("nothing to save: call adapt() first")
        model, _ = self._require_model()
        from safetensors.torch import save_file

        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = set(self.adapter["trainable_names"])
        tensors = {k: v.detach().cpu().contiguous() for k, v in model.state_dict().items() if k in names}
        weights_path = out / ARTIFACT_WEIGHTS_NAME
        save_file(tensors, str(weights_path), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "format_version": ARTIFACT_FORMAT_VERSION,
            "base_model": {
                "id": MODEL_ID,
                "revision": MODEL_REVISION,
                "key": MODEL_KEY,
                "weight_file": WEIGHT_FILE,
                "weight_sha256": WEIGHT_SHA256,
            },
            "adapter": {k: v for k, v in self.adapter.items() if k not in ("history", "trainable_names")},
            "history": self.adapter["history"],
            "tensors": sorted(tensors),
            "files": [
                {
                    "path": ARTIFACT_WEIGHTS_NAME,
                    "bytes": weights_path.stat().st_size,
                    "sha256": _sha256(weights_path),
                }
            ],
            "metadata": dict(metadata or {}),
        }
        (out / ARTIFACT_MANIFEST_NAME).write_text(
            json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
        )
        return out

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Verify an adapter's manifest and digest, then overwrite exactly the tensors it carries."""
        root = Path(artifact_dir)
        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding="utf-8"))
        if manifest.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
        base = manifest.get("base_model", {})
        if (base.get("id"), base.get("revision"), base.get("weight_sha256")) != (
            MODEL_ID,
            MODEL_REVISION,
            WEIGHT_SHA256,
        ):
            raise ValueError("artifact was adapted from a different base model, revision or weight file")
        entry = manifest["files"][0]
        weights_path = root / entry["path"]
        if not weights_path.is_file():
            raise FileNotFoundError(f"artifact weights missing: {weights_path}")
        if _sha256(weights_path) != entry["sha256"] or weights_path.stat().st_size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: digest or size mismatch; refusing to load")
        model, _ = self._require_model()
        from safetensors.torch import load_file

        tensors = load_file(str(weights_path))
        if sorted(tensors) != manifest["tensors"]:
            raise ValueError("artifact tensor names differ from its manifest")
        state = model.state_dict()
        for key, value in tensors.items():
            if key not in state or not (
                key.startswith("model.decoder.layers.") or key.startswith("classification_head.")
            ):
                raise ValueError(
                    f"artifact tensor {key} is not an adaptable decoder or head tensor of the base model"
                )
            if tuple(value.shape) != tuple(state[key].shape):
                raise ValueError(
                    f"artifact tensor {key} has shape {tuple(value.shape)}, "
                    f"base has {tuple(state[key].shape)}"
                )
        merged = dict(state)
        merged.update({k: v.to(state[k].dtype) for k, v in tensors.items()})
        model.load_state_dict(merged, strict=True)
        model.eval()
        self.adapter = {
            **manifest["adapter"],
            "trainable_names": manifest["tensors"],
            "history": manifest.get("history", []),
        }
        return manifest

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> BARTZeroShotClassificationPipeline:
        pipeline = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)
        pipeline.load_artifact(artifact_dir)
        return pipeline

**Module 3/3:** `src/bart_zero_shot_classification_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Labelled-text dataset contract for adapting the zero-shot classifier: the pinned Banking77 sample,
validation, seeded splitting, BYOD loaders and CSV export.

The default dataset is **real** and out of the NLI model's domain: Banking77 (Casanueva et al., 2020;
CC BY 4.0), 13,083 customer-support queries labelled with 77 fine-grained banking intents. Two CSV files
(`train.csv`, `test.csv`) are fetched from the PolyAI `task-specific-datasets` repository at a pinned
commit and refused on any byte-size or SHA-256 mismatch. The tutorial keeps ten intents (`LABEL_SET`) with
a readable phrase per intent so the zero-shot hypothesis reads as English ("This customer message is about a
lost or stolen card."); training and validation queries are drawn from `train.csv`, test queries from
`test.csv` — the release's own partition.

A record is ``{id, text, label}``: a customer message and the phrase of its gold intent.
"""

from __future__ import annotations

import csv
import hashlib
import io
import json
import random
import re
import urllib.request
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

# standalone rewrite (build_notebook.py): `from .pipeline import MAX_LABEL_CHARS, MAX_LABELS, MAX_TEXT_CHARS, MODEL_ID` removed — names are kernel globals defined by the carried modules

CORPUS_NAME = "Banking77"
CORPUS_RELEASE = "PolyAI-LDN/task-specific-datasets @ 57ec275d8078af65b7731c2a98be812d844a6d6b"
CORPUS_BASE_URL = (
    "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/"
    "57ec275d8078af65b7731c2a98be812d844a6d6b/banking_data/"
)
CORPUS_FILES = {
    "train": ("train.csv", 839_073, "b06e26ac675513959a63135f11b94ea7786ed02da65db93a5650d8838cbc664b"),
    "test": ("test.csv", 239_961, "d12d6e3bc4c3103966ae786dc435913c0c563dfa328f5a3646d0e62cfeeb474d"),
}
CORPUS_LICENSE = "CC BY 4.0 (Casanueva et al. 2020; PolyAI-LDN/task-specific-datasets)"
CORPUS_ROWS = {"train": 10_003, "test": 3_080}
CORPUS_INTENTS = 77
DEFAULT_CACHE_DIR = Path("weights") / "banking77"
# The ten intents the tutorial keeps, each with the phrase the hypothesis template receives.
LABEL_SET: dict[str, str] = {
    "card_arrival": "card arrival",
    "lost_or_stolen_card": "a lost or stolen card",
    "exchange_rate": "the exchange rate",
    "change_pin": "changing the PIN",
    "declined_card_payment": "a declined card payment",
    "transfer_not_received_by_recipient": "a transfer not received by the recipient",
    "atm_support": "ATM support",
    "age_limit": "the age limit",
    "terminate_account": "closing the account",
    "top_up_failed": "a failed top-up",
}
DEFAULT_HYPOTHESIS_TEMPLATE = "This customer message is about {}."
SAMPLE_SEED = 42
SAMPLE_SPLIT = {"train": 400, "validation": 100, "test": 200}  # balanced over the ten intents
MIN_RECORDS = 8
MAX_RECORDS = 20_000
MIN_LABEL_KINDS = 2
_ID_RE = re.compile(r"^[A-Za-z0-9_.:-]{1,64}$")


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def fetch_corpus(*, cache_dir: str | Path | None = None, fetcher: Any = None) -> dict[str, bytes]:
    """Return the two pinned Banking77 CSV files (bytes) from the cache or the repository, verified."""
    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    cache.mkdir(parents=True, exist_ok=True)
    out = {}
    for split, (name, size, digest) in CORPUS_FILES.items():
        local = cache / name
        data = local.read_bytes() if local.is_file() else b""
        if len(data) != size or _sha256_bytes(data) != digest:
            url = CORPUS_BASE_URL + name
            if fetcher is not None:
                data = fetcher(url)
            else:
                with urllib.request.urlopen(url, timeout=120) as response:  # noqa: S310 (pinned https URL)
                    data = response.read()
            if len(data) != size or _sha256_bytes(data) != digest:
                raise ValueError(
                    f"{name}: fetched {len(data)} bytes with sha256 {_sha256_bytes(data)[:16]}…, "
                    f"pinned {size} / {digest[:16]}…"
                )
            local.write_bytes(data)
        out[split] = data
    return out


def read_corpus(files: Mapping[str, bytes]) -> dict[str, list[dict[str, Any]]]:
    """Parse the CSV members (columns `text`, `category`) into flat records keeping the raw intent name."""
    out = {}
    for split in CORPUS_FILES:
        if split not in files:
            raise ValueError(f"corpus is missing the {split} file")
        rows = list(csv.DictReader(io.StringIO(files[split].decode("utf-8"))))
        if not rows or {"text", "category"} - set(rows[0]):
            raise ValueError(f"{split}: expected columns text and category")
        if len(rows) != CORPUS_ROWS[split]:
            raise ValueError(f"{split}: {len(rows)} rows, expected {CORPUS_ROWS[split]}")
        out[split] = [
            {"id": f"{split}-{i:05d}", "text": r["text"].strip(), "intent": r["category"].strip()}
            for i, r in enumerate(rows)
        ]
        intents = {r["intent"] for r in out[split]}
        if len(intents) != CORPUS_INTENTS:
            raise ValueError(f"{split}: {len(intents)} intents, expected {CORPUS_INTENTS}")
    return out


def filter_records(
    records: Sequence[Mapping[str, Any]], *, label_set: Mapping[str, str] | None = None
) -> list[dict[str, Any]]:
    """Keep records whose intent is in the label set, mapped to its phrase; drop repeated texts."""
    label_set = dict(label_set or LABEL_SET)
    seen: set[str] = set()
    kept = []
    for record in records:
        intent = str(record.get("intent", record.get("label", "")))
        if intent not in label_set:
            continue
        text = str(record["text"]).strip()
        key = text.lower()
        if not text or key in seen or len(text) > MAX_TEXT_CHARS:
            continue
        seen.add(key)
        kept.append({"id": record["id"], "text": text, "label": label_set[intent], "intent": intent})
    return kept


def build_sample_dataset(
    corpus: Mapping[str, Sequence[Mapping[str, Any]]],
    *,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
    label_set: Mapping[str, str] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """Balanced seeded draws: training and validation from `train` (disjoint texts), test from `test`."""
    sizes = dict(sizes or SAMPLE_SPLIT)
    label_set = dict(label_set or LABEL_SET)
    n_labels = len(label_set)
    for name, size in sizes.items():
        if size % n_labels:
            raise ValueError(f"{name} size {size} is not a multiple of the {n_labels} labels")
    rng = random.Random(seed)
    pools = {
        "train": filter_records(corpus["train"], label_set=label_set),
        "test": filter_records(corpus["test"], label_set=label_set),
    }
    by_label = {
        split: {phrase: [r for r in pool if r["label"] == phrase] for phrase in label_set.values()}
        for split, pool in pools.items()
    }
    for split in by_label.values():
        for records in split.values():
            rng.shuffle(records)
    cursor = {phrase: 0 for phrase in label_set.values()}
    out: dict[str, list[dict[str, Any]]] = {}
    for name, size in sizes.items():
        source = "test" if name == "test" else "train"
        per_label = size // n_labels
        picked = []
        for phrase in label_set.values():
            pool = by_label[source][phrase]
            start = cursor[phrase] if source == "train" else 0
            chunk = pool[start : start + per_label]
            if len(chunk) < per_label:
                raise ValueError(
                    f"{name}: only {len(chunk)} records available for {phrase!r}, need {per_label}"
                )
            picked.extend(chunk)
            if source == "train":
                cursor[phrase] = start + per_label
        rng.shuffle(picked)
        out[name] = [
            {"id": f"{name}-{i:04d}", "text": r["text"], "label": r["label"], "intent": r["intent"]}
            for i, r in enumerate(picked)
        ]
    return out


def fetch_sample_dataset(
    *,
    cache_dir: str | Path | None = None,
    fetcher: Any = None,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """The tutorial splits from the pinned corpus."""
    return build_sample_dataset(
        read_corpus(fetch_corpus(cache_dir=cache_dir, fetcher=fetcher)), seed=seed, sizes=sizes
    )


def _check_record(record: Any, index: int) -> dict[str, Any]:
    label_name = f"records[{index}]"
    if not isinstance(record, Mapping):
        raise ValueError(f"{label_name} must be a mapping with id/text/label")
    for key in ("id", "text", "label"):
        if key not in record:
            raise ValueError(f"{label_name} is missing {key!r}")
    rid, text, label = record["id"], record["text"], record["label"]
    if not isinstance(rid, str) or not _ID_RE.match(rid):
        raise ValueError(f"{label_name}: id must match {_ID_RE.pattern}")
    if not isinstance(text, str):
        raise ValueError(f"{label_name}: text must be a string")
    if not text.strip():
        raise ValueError(f"{label_name}: text is empty")
    if len(text) > MAX_TEXT_CHARS:
        raise ValueError(
            f"{label_name}: text has {len(text)} chars; ceiling is MAX_TEXT_CHARS={MAX_TEXT_CHARS}"
        )
    if not isinstance(label, str) or not label.strip():
        raise ValueError(f"{label_name}: label must be a non-empty string")
    if len(label) > MAX_LABEL_CHARS:
        raise ValueError(
            f"{label_name}: label has {len(label)} chars; ceiling is MAX_LABEL_CHARS={MAX_LABEL_CHARS}"
        )
    item = {"id": rid, "text": text.strip(), "label": label.strip()}
    if "intent" in record:
        item["intent"] = str(record["intent"])
    return item


def validate_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    min_records: int = MIN_RECORDS,
    max_records: int = MAX_RECORDS,
    labels: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Structural validation of a labelled-text dataset; raises ValueError before any model import.
    With `labels`, every record's label must be one of them."""
    if isinstance(records, Mapping) or not isinstance(records, Sequence) or isinstance(records, (str, bytes)):
        raise ValueError("records must be a list of {id, text, label} mappings")
    if not min_records <= len(records) <= max_records:
        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")
    allowed = set(labels) if labels is not None else None
    checked = []
    ids: set[str] = set()
    texts: set[str] = set()
    counts: dict[str, int] = {}
    for index, record in enumerate(records):
        item = _check_record(record, index)
        if item["id"] in ids:
            raise ValueError(f"duplicate id {item['id']!r}")
        if allowed is not None and item["label"] not in allowed:
            raise ValueError(f"records[{index}]: label {item['label']!r} is not in the label set")
        ids.add(item["id"])
        texts.add(item["text"].lower())
        counts[item["label"]] = counts.get(item["label"], 0) + 1
        checked.append(item)
    if len(counts) > MAX_LABELS:
        raise ValueError(f"{len(counts)} distinct labels; ceiling is MAX_LABELS={MAX_LABELS}")
    return {
        "records": checked,
        "n_records": len(checked),
        "unique_texts": len(texts),
        "labels": sorted(counts),
        "label_counts": dict(sorted(counts.items())),
        "text_chars": {
            "min": min(len(r["text"]) for r in checked),
            "max": max(len(r["text"]) for r in checked),
        },
        "digest": dataset_digest(checked),
        "model_id": MODEL_ID,
    }


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    payload = [[r["id"], r["text"], r["label"]] for r in records]
    return _sha256_bytes(json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode("utf-8"))


def label_names(records: Sequence[Mapping[str, Any]]) -> list[str]:
    """The sorted label vocabulary of a dataset (the `labels` argument for `classify`)."""
    names = sorted({str(r["label"]) for r in records})
    if len(names) < MIN_LABEL_KINDS:
        raise ValueError(f"a dataset needs at least {MIN_LABEL_KINDS} distinct labels")
    return names


def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Assert no lower-cased text appears in two splits (leakage check)."""
    seen: dict[str, str] = {}
    for name, records in splits.items():
        for record in records:
            key = str(record["text"]).lower()
            if key in seen and seen[key] != name:
                raise ValueError(f"text {record['text'][:60]!r} appears in both {seen[key]} and {name}")
            seen[key] = name
    return {name: len(records) for name, records in splits.items()}


def split_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    val_fraction: float = 0.15,
    test_fraction: float = 0.2,
    seed: int = 0,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded stratified split of a BYOD dataset into train/validation/test after de-duplicating texts."""
    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")
    checked = validate_dataset(records)["records"]
    seen: set[str] = set()
    by_label: dict[str, list[dict[str, Any]]] = {}
    for record in checked:
        key = record["text"].lower()
        if key not in seen:
            seen.add(key)
            by_label.setdefault(record["label"], []).append(record)
    rng = random.Random(seed)
    splits: dict[str, list[dict[str, Any]]] = {"test": [], "validation": [], "train": []}
    for label in sorted(by_label):
        group = by_label[label]
        rng.shuffle(group)
        n_test = max(1, round(len(group) * test_fraction))
        n_val = round(len(group) * val_fraction)
        splits["test"].extend(group[:n_test])
        splits["validation"].extend(group[n_test : n_test + n_val])
        splits["train"].extend(group[n_test + n_val :])
    for part in splits.values():
        rng.shuffle(part)
    if len(splits["train"]) < MIN_RECORDS:
        raise ValueError(
            f"split leaves {len(splits['train'])} training records; at least {MIN_RECORDS} are required"
        )
    return splits


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    """Read `{id, text, label}` records from CSV (columns id, text, label), a JSON array or JSONL."""
    file_path = Path(path)
    if not file_path.is_file():
        raise FileNotFoundError(f"dataset not found: {file_path}")
    suffix = file_path.suffix.lower()
    text = file_path.read_text(encoding="utf-8")
    if suffix == ".csv":
        rows = list(csv.DictReader(io.StringIO(text)))
        missing = {"id", "text", "label"} - set(rows[0].keys() if rows else set())
        if missing:
            raise ValueError(f"CSV is missing columns {sorted(missing)}")
        return [{"id": r["id"], "text": r["text"], "label": r["label"]} for r in rows]
    if suffix == ".jsonl":
        return [json.loads(line) for line in text.splitlines() if line.strip()]
    if suffix == ".json":
        data = json.loads(text)
        if not isinstance(data, list):
            raise ValueError("JSON dataset must be an array of records")
        return data
    raise ValueError("BYOD datasets must be .csv, .json or .jsonl")


def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=["id", "text", "label"])
        writer.writeheader()
        for record in records:
            writer.writerow({"id": record["id"], "text": record["text"], "label": record["label"]})
    return out

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `7`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `d7645e127eaf…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `BARTZeroShotClassificationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "bart-large-mnli",
  "modelId": "facebook/bart-large-mnli",
  "revision": "d7645e127eaf1aefc7862fd59a17a5aa8558b8ce",
  "files": [
    {
      "path": "README.md",
      "bytes": 3793,
      "sha256": "d022b7cb54ca2f65fac41cf6c9601b4491b4ac22a301227288a38beef2a18aeb"
    },
    {
      "path": "config.json",
      "bytes": 1154,
      "sha256": "a0f9bcb245b680a96ccae0ad8d155f267ec3e3c971ef4a4937e52ea9ba368a86"
    },
    {
      "path": "merges.txt",
      "bytes": 456318,
      "sha256": "1ce1664773c50f3e0cc8842619a93edc4624525b728b188a9e0be33b7726adc5"
    },
    {
      "path": "model.safetensors",
      "bytes": 1629437147,
      "sha256": "cfbb687dbbd9df99fe865e1860350a22aebac4d26ee4bcb50217f1df606a018e"
    },
    {
      "path": "tokenizer.json",
      "bytes": 1355863,
      "sha256": "847bbeab6174d66a88898f729d52fa8d355fafe1bea101cf960dd404581df70e"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 26,
      "sha256": "5e04eb606e3a1583530a42e36c2a6b6615c86f34fe77e44d9ddeb43ff940931f"
    },
    {
      "path": "vocab.json",
      "bytes": 898822,
      "sha256": "06b4d46c8e752d410213d9548eb27a54db70fda0319b6271fb8d59dead5e1cab"
    }
  ],
  "totalBytes": 1632153123
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = BARTZeroShotClassificationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Labelled corpus, validation and split

`fetch_corpus` downloads the two pinned Banking77 CSV files (or reads them from the cache), refuses a byte-size or SHA-256 mismatch per file before it is parsed, and `read_corpus` checks the columns, the row counts and the 77 intents. `build_sample_dataset` keeps the ten intents in `LABEL_SET`, maps each to its readable phrase, drops repeated messages, and draws **balanced** seeded samples — 40 training and 10 validation messages per intent from `train.csv` (disjoint), 20 test messages per intent from `test.csv` — the release's own partition. `validate_dataset` then checks every record against the contract and reports the label counts, `label_names` derives the ten-phrase label set every `classify` call will score, `check_split_disjoint` asserts no message appears in two splits, and the training split is written to `outputs/bart_zero_shot_classification_train.csv` in the shape BYOD expects.

Look for: 10,003 + 3,080 raw rows, three digests, splits 400 / 100 / 200 with 40 / 10 / 20 per label, and four refusal probes — a duplicate id, a label outside the label set, a missing field and a dataset too small to split — each rejected before `torch` does anything.

In [ ]:
import hashlib
import io
import json

USE_BYOD = False  # @param {type:"boolean"}
SPLIT_SEED = 42  # @param {type:"integer"}
TEMPLATE = 'This customer message is about {}.'  # @param {type:"string"}

os.makedirs('outputs', exist_ok=True)
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    file_name, payload = next(iter(uploaded.items()))
    byod_path = Path('work') / file_name
    byod_path.parent.mkdir(parents=True, exist_ok=True)
    byod_path.write_bytes(payload)
    records = load_byod_dataset(byod_path)
    splits = split_dataset(records, seed=SPLIT_SEED)
    data_source = 'BYOD (' + file_name + ')'
    raw_rows = {'byod': len(records)}
else:
    corpus = read_corpus(fetch_corpus(cache_dir='weights/banking77'))
    raw_rows = {name: len(part) for name, part in corpus.items()}
    splits = build_sample_dataset(corpus, seed=SPLIT_SEED)
    data_source = f'{CORPUS_NAME} ({CORPUS_RELEASE}; {CORPUS_LICENSE}), {len(LABEL_SET)} of {CORPUS_INTENTS} intents'
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']
dataset_manifests = {name: validate_dataset(part) for name, part in splits.items()}
labels = label_names(train_records)
disjoint = check_split_disjoint(splits)
write_dataset_csv(train_records, 'outputs/bart_zero_shot_classification_train.csv')
print({'data_source': data_source, 'raw_rows': raw_rows, 'splits': disjoint, 'labels': labels, 'file_sha256': {k: v[2][:12] + '...' for k, v in CORPUS_FILES.items()}})
for name, manifest in dataset_manifests.items():
    print({name: {'n': manifest['n_records'], 'unique_texts': manifest['unique_texts'], 'label_counts': manifest['label_counts'], 'text_chars': manifest['text_chars'], 'digest': manifest['digest'][:16] + '...'}})
print({'example': {k: train_records[0][k] for k in ('id', 'text', 'label')}})

probes = {
    'duplicate id': [{**r, 'id': 'same'} for r in train_records[:8]],
    'label outside the label set': ([{**train_records[0], 'label': 'something else'}, *train_records[1:8]], labels),
    'missing field': [{'id': r['id'], 'text': r['text']} for r in train_records[:8]],
    'too small': train_records[:3],
}
for name, probe in probes.items():
    try:
        validate_dataset(*probe) if isinstance(probe, tuple) else validate_dataset(probe)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

## 5. Classify through the inference contract

Before any adaptation, the inference contract is exercised as it always was, on three synthetic sentences and a three-label set authored in this cell (the upstream README's `one day I will see the world` and two more). `validate_inputs` applies exactly the checks `classify` applies — text types and character ceilings, 1..`MAX_LABELS` unique labels under `MAX_LABEL_CHARS`, one `{}` in the template — and returns an input manifest; the pair-token ceiling `MAX_TEXT_TOKENS` needs the real tokenizer and is enforced inside `classify`, which **rejects with a `ValueError` naming the count, never truncates**. A duplicate-label probe is validated too and its rejection recorded as a finding. `classify` returns one entry per label ordered by descending `score` with the raw entailment and contradiction logits, plus `top_label`, `multi_label`, the template, the `decision_rule`, `n_tokens` and the model identity. **Score semantics:** with `multi_label=False` the scores sum to one over the label set and the **default decision rule is `argmax`**, so a text that fits none of the labels still gets a winner; with `multi_label=True` each score is that label's own entailment probability and they do not sum to one. In neither mode is the score a calibrated probability of class membership. Whether the top labels are *right* is what Section 6 measures on 200 gold-labelled messages, not what three authored sentences can tell you.

In [ ]:
import time

MULTI_LABEL = False  # @param {type:"boolean"}

demo_labels = ['travel', 'cooking', 'dancing']
texts = ['one day I will see the world', 'Whisk the eggs and fold in the flour before baking.', 'The tango class meets every Thursday evening.']
expected = ['travel', 'cooking', 'dancing']
text_ids = [f't{index + 1}' for index in range(len(texts))]
ceilings = {'MAX_TEXT_CHARS': MAX_TEXT_CHARS, 'MAX_TEXT_TOKENS': MAX_TEXT_TOKENS, 'MAX_LABELS': MAX_LABELS, 'MAX_LABEL_CHARS': MAX_LABEL_CHARS, 'HYPOTHESIS_TEMPLATE': HYPOTHESIS_TEMPLATE, 'NUM_NLI_LABELS': NUM_NLI_LABELS, 'ENTAILMENT_INDEX': ENTAILMENT_INDEX, 'CONTRADICTION_INDEX': CONTRADICTION_INDEX}
print(ceilings)
print({'decision_rules': {'single': DECISION_RULE_SINGLE, 'multi': DECISION_RULE_MULTI}})
input_manifest = validate_inputs(texts, demo_labels, multi_label=MULTI_LABEL, names=text_ids)
try:
    validate_inputs(texts, [*demo_labels, demo_labels[0]], multi_label=MULTI_LABEL)
except ValueError as exc:
    input_manifest['findings'].append({'input': 'duplicate-label-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/bart_zero_shot_classification_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
results = []
for text_id, text in zip(text_ids, texts, strict=True):
    started = time.perf_counter()
    result = pipe.classify(text, demo_labels, multi_label=MULTI_LABEL)
    elapsed = time.perf_counter() - started
    results.append(result)
    scores = [entry['score'] for entry in result['labels']]
    checks = {
        'one_entry_per_label': sorted(entry['label'] for entry in result['labels']) == sorted(demo_labels),
        'scores_descending': all(a >= b for a, b in zip(scores, scores[1:], strict=False)),
        'scores_in_unit_interval': all(0.0 <= s <= 1.0 for s in scores),
        'single_label_scores_sum_to_one': MULTI_LABEL or abs(sum(scores) - 1.0) < 1e-6,
        'top_label_is_first': result['top_label'] == result['labels'][0]['label'],
        'n_tokens_within_ceiling': 1 <= result['n_tokens'] <= MAX_TEXT_TOKENS,
    }
    if not all(checks.values()):
        raise RuntimeError(f'classify output for {text_id} failed a sanity check: {checks}')
    print({'id': text_id, 'text': text[:60], 'top_label': result['top_label'], 'scores': {e['label']: round(e['score'], 4) for e in result['labels']}, 'seconds': round(elapsed, 3), 'n_tokens': result['n_tokens'], 'checks': checks})
other_mode = pipe.classify(texts[0], demo_labels, multi_label=not MULTI_LABEL)
print({'first_text_other_mode': {'multi_label': other_mode['multi_label'], 'scores': {e['label']: round(e['score'], 4) for e in other_mode['labels']}}})
print({'top_label_matches_expected': [r['top_label'] == g for r, g in zip(results, expected, strict=True)], 'findings': len(input_manifest['findings'])})

## 6. Baseline and the frozen zero-shot model's score on the test split

Two numbers frame the adaptation. The **majority-class baseline** predicts the most frequent gold label for every message — on a balanced ten-label split that is exactly 10 % accuracy, the floor any classifier must clear. The **frozen zero-shot model** classifies the 200 test messages over the ten phrases with the template from Section 4 and is scored with the same two metrics: **accuracy** (exact top-label match) and **macro-F1** (the unweighted mean of the per-label F1, so a rarely predicted label counts as much as a popular one). Expect the frozen model to do well already — these intents have readable names — and read the per-label F1 to see which phrases the NLI model reads badly (the build record found `ATM support` and `card arrival` far below the rest). About one minute on CPU.

In [ ]:
baseline_majority = majority_baseline(test_records)
print({'majority_baseline': {'accuracy': round(baseline_majority['accuracy'], 2), 'macro_f1': round(baseline_majority['macro_f1'], 2), 'n': baseline_majority['n'], 'rule': baseline_majority['baseline']}})
t0 = time.perf_counter()
frozen_test = pipe.evaluate(test_records, labels, hypothesis_template=TEMPLATE)
print({'frozen_model_test': {'accuracy': round(frozen_test['accuracy'], 2), 'macro_f1': round(frozen_test['macro_f1'], 2), 'n': frozen_test['n'], 'n_labels': frozen_test['n_labels'], 'verdict': frozen_test['verdict']}, 'seconds': round(time.perf_counter() - t0, 1)})
print({'per_label_f1': {label: round(v['f1'], 1) for label, v in frozen_test['per_label'].items()}})
print({'definitions': frozen_test['definitions']})
for record in test_records[:2]:
    item = pipe.classify(record['text'], labels, hypothesis_template=TEMPLATE)
    print({'text': record['text'][:80], 'frozen': item['top_label'], 'score': round(item['labels'][0]['score'], 3), 'gold': record['label']})
assert frozen_test['accuracy'] > baseline_majority['accuracy']

## 7. Bounded fine-tuning of the last decoder blocks and the NLI head

`pipe.adapt` turns every training message into two NLI pairs — (message, template(gold phrase)) labelled *entailment* and (message, template(a seeded wrong phrase)) labelled *contradiction* — and trains only the last `TRAINABLE_DECODER_LAYERS` decoder blocks plus the classification head: two blocks by default, 34,646,019 of 407,344,131 parameters; the encoder, the shared embeddings and the earlier decoder blocks stay frozen. Cross-entropy over the three NLI logits, AdamW at a fixed learning rate, gradient clipping at 1.0, seeded shuffling and no scheduler; pairs are truncated to 256 BPE tokens **during training only**. Epoch 0 records the frozen model's validation accuracy and macro-F1; every epoch is scored on the validation split with the same labels and template, and the epoch with the highest validation accuracy is kept.

Watch validation accuracy climb into the high nineties over two epochs (about a minute of training plus a validation pass per epoch on CPU). The adapted model is still an NLI scorer: it answers any label set, but it has been pulled towards these ten phrases and this template.

In [ ]:
EPOCHS = 2  # @param {type:"integer"}
LEARNING_RATE = 2e-5  # @param {type:"number"}
BATCH_SIZE = 16  # @param {type:"integer"}
TRAINABLE_DECODER_LAYERS = 2  # @param {type:"integer"}

def report(entry):
    row = {'epoch': entry['epoch'], 'train_loss': None if entry['train_loss'] is None else round(entry['train_loss'], 4)}
    if entry.get('val'):
        row['val_accuracy'] = round(entry['val']['accuracy'], 2)
        row['val_macro_f1'] = round(entry['val']['macro_f1'], 2)
    if 'note' in entry:
        row['note'] = entry['note']
    print(row)

t0 = time.perf_counter()
adapt_result = pipe.adapt(train_records, val_records, labels=labels, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, trainable_decoder_layers=TRAINABLE_DECODER_LAYERS, hypothesis_template=TEMPLATE, progress=report)
adapt_seconds = round(time.perf_counter() - t0, 1)
print({'trainable_parameters': adapt_result['n_trainable'], 'total_parameters': adapt_result['n_total'], 'pairs_per_record': adapt_result['pairs_per_record'], 'best_epoch': adapt_result['best_epoch'], 'selection': adapt_result['selection'], 'seconds': adapt_seconds})

## 8. Held-out evaluation

The test split was never used for training or epoch selection, and no message in it appears in the training or validation splits. The adapted model is scored exactly as the frozen model was in Section 6, and the three numbers are put side by side with the per-label F1 before and after. Look for an accuracy gain of ten points or more — the cell asserts the adapted accuracy is above the frozen accuracy — and for the weakest phrases of Section 6 recovering. Two hundred messages from one seeded split of one corpus give no dispersion estimate; the deltas are sample-sanity evidence that the adaptation contract works, not a benchmark, and a gain on ten banking intents says nothing about your label set until you measure it there.

In [ ]:
adapted_test = pipe.evaluate(test_records, labels, hypothesis_template=TEMPLATE)
adapted_val = pipe.evaluate(val_records, labels, hypothesis_template=TEMPLATE)
comparison = {
    'accuracy': {'majority': round(baseline_majority['accuracy'], 2), 'frozen': round(frozen_test['accuracy'], 2), 'adapted': round(adapted_test['accuracy'], 2)},
    'macro_f1': {'majority': round(baseline_majority['macro_f1'], 2), 'frozen': round(frozen_test['macro_f1'], 2), 'adapted': round(adapted_test['macro_f1'], 2)},
    'delta_vs_frozen': {'accuracy': round(adapted_test['accuracy'] - frozen_test['accuracy'], 2), 'macro_f1': round(adapted_test['macro_f1'] - frozen_test['macro_f1'], 2)},
    'per_label_f1': {label: {'frozen': round(frozen_test['per_label'][label]['f1'], 1), 'adapted': round(adapted_test['per_label'][label]['f1'], 1)} for label in labels},
}
for metric, row in comparison.items():
    print({metric: row})
for record in test_records[:2]:
    item = pipe.classify(record['text'], labels, hypothesis_template=TEMPLATE)
    print({'text': record['text'][:80], 'adapted': item['top_label'], 'score': round(item['labels'][0]['score'], 3), 'gold': record['label']})
evaluation_report_payload = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': MODEL_KEY},
    'data_source': data_source,
    'dataset_digests': {name: manifest['digest'] for name, manifest in dataset_manifests.items()},
    'splits': disjoint,
    'labels': labels,
    'hypothesis_template': TEMPLATE,
    'baselines': {'majority': baseline_majority},
    'frozen_test': frozen_test,
    'validation_metrics': adapted_val,
    'test_metrics': adapted_test,
    'comparison': comparison,
    'adaptation': {k: v for k, v in adapt_result.items() if k not in ('history', 'trainable_names')},
    'history': adapt_result['history'],
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/bart_zero_shot_classification_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report_payload, f, indent=2, ensure_ascii=False)
assert adapted_test['accuracy'] > frozen_test['accuracy']
print({'report': 'outputs/bart_zero_shot_classification_evaluation_report.json'})

## 9. Classify new messages, export the adapter and reload it

Ten messages that were in none of the splits (one per intent, drawn from the training file beyond the sample) are classified by the adapted model through the same `classify` contract as Section 5 and scored with the batch `evaluation_report` — the inference-stage helper, whose `accuracy` on ten gold-labelled items is a `sample-sanity` observation, never a benchmark — and with `pipe.evaluate` (`measured-small-sample`).

`pipe.save_artifact` writes the trained tensors — the last two decoder blocks and the NLI head, about 139 MB — as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the digest of the base `model.safetensors`, the tensor names, the file size and SHA-256, the training configuration (labels, template, hyperparameters) and the epoch history (OUT8). `BARTZeroShotClassificationPipeline.from_artifact` re-verifies the base snapshot, checks the artifact manifest and digest **before** deserialising, refuses any tensor that is not an adaptable decoder or head tensor, and overlays the tensors onto a freshly loaded base — a new object from files, not the in-memory model (VER2). The cell asserts identical top labels (VER4).

In [ ]:
import csv
import shutil

if USE_BYOD:
    new_records = [{**r, 'id': f'new-{i:02d}'} for i, r in enumerate(test_records[:10])]
else:
    used = {r['text'].lower() for part in splits.values() for r in part}
    spare = [r for r in filter_records(corpus['train']) if r['text'].lower() not in used]
    new_records = [{**next(r for r in spare if r['label'] == label), 'id': f'new-{i:02d}'} for i, label in enumerate(labels)]
new_results = [pipe.classify(r['text'], labels, hypothesis_template=TEMPLATE) for r in new_records]
new_report = evaluation_report(new_results, [r['label'] for r in new_records], sample_kind='ten unseen Banking77 messages' if not USE_BYOD else 'BYOD test records')
new_metrics = pipe.evaluate(new_records, labels, hypothesis_template=TEMPLATE)
for record, item in zip(new_records, new_results, strict=True):
    print({'id': record['id'], 'text': record['text'][:70], 'adapted': item['top_label'], 'score': round(item['labels'][0]['score'], 3), 'gold': record['label']})
print({'new_messages': {'verdict': new_report['verdict'], 'accuracy': new_report['metrics'][0]['value'], 'reason': new_report['reason']}, 'evaluate_verdict': new_metrics['verdict'], 'macro_f1': round(new_metrics['macro_f1'], 2)})
with open('outputs/bart_zero_shot_classification_predictions.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=['id', 'text', 'top_label', 'score', 'gold', 'n_tokens'])
    writer.writeheader()
    for record, item in zip(new_records, new_results, strict=True):
        writer.writerow({'id': record['id'], 'text': record['text'], 'top_label': item['top_label'], 'score': item['labels'][0]['score'], 'gold': record['label'], 'n_tokens': item['n_tokens']})

artifact_dir = Path('outputs/bart_zero_shot_classification_adapter')
shutil.rmtree(artifact_dir, ignore_errors=True)
pipe.save_artifact(artifact_dir, metadata={'tutorial': 'bart_zero_shot_classification', 'data_source': data_source})
artifact_manifest = json.loads((artifact_dir / 'manifest.json').read_text(encoding='utf-8'))
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'tensors': len(artifact_manifest['tensors']), 'bytes': artifact_manifest['files'][0]['bytes'], 'sha256': artifact_manifest['files'][0]['sha256'][:16] + '...'})

reloaded = BARTZeroShotClassificationPipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR, device=pipe.device)
before = [pipe.classify(r['text'], labels, hypothesis_template=TEMPLATE)['top_label'] for r in test_records[:8]]
after = [reloaded.classify(r['text'], labels, hypothesis_template=TEMPLATE)['top_label'] for r in test_records[:8]]
parity = {'identical_labels': sum(a == b for a, b in zip(before, after, strict=True)), 'of': len(before)}
print({'reload_parity': parity, 'reloaded_best_epoch': reloaded.adapter['best_epoch']})
assert parity['identical_labels'] == parity['of']

weight_entry = next(entry for entry in snapshot['files'] if entry['path'] == WEIGHT_FILE)
result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'snapshot': {'path': str(WEIGHTS_DIR), 'files': len(snapshot['files']), 'total_bytes': snapshot.get('totalBytes'), 'fetched_this_run': fetched, 'weight_file': WEIGHT_FILE, 'weight_format': 'safetensors, digest-verified', 'weight_sha256': weight_entry['sha256']},
    'data_source': data_source,
    'corpus': {'name': CORPUS_NAME, 'release': CORPUS_RELEASE, 'base_url': CORPUS_BASE_URL, 'files': {k: {'name': v[0], 'bytes': v[1], 'sha256': v[2]} for k, v in CORPUS_FILES.items()}, 'license': CORPUS_LICENSE, 'label_set': LABEL_SET},
    'inference_contract': {'input_manifest': input_manifest, 'demo_results': [{'id': i, 'top_label': r['top_label'], 'scores': {e['label']: e['score'] for e in r['labels']}, 'n_tokens': r['n_tokens'], 'decision_rule': r['decision_rule']} for i, r in zip(text_ids, results, strict=True)], 'expected': expected},
    'labels': labels,
    'hypothesis_template': TEMPLATE,
    'comparison': comparison,
    'new_messages': new_report,
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['files'][0]['sha256'], 'bytes': artifact_manifest['files'][0]['bytes'], 'tensors': len(artifact_manifest['tensors'])},
    'reload_parity': parity,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'device': pipe.device, 'dtype': 'float32', 'source': pipe.source},
}
with open('outputs/bart_zero_shot_classification_result.json', 'w', encoding='utf-8') as handle:
    json.dump(result_payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The frozen zero-shot model already reads most of the ten banking intents from their phrases — its test accuracy sits far above the 10 % majority floor — and a bounded fine-tuning of the last two decoder blocks and the NLI head on 800 entailment/contradiction pairs built from 400 messages lifts held-out accuracy and macro-F1 by more than ten points in a few minutes on CPU, recovering the phrases the frozen model read badly, with a 139 MB adapter that reloads to identical labels. That is the claim: the adaptation contract works end to end on a real labelled corpus, and the numbers it produces are read against the majority baseline and the frozen model rather than in isolation.

The test split is 200 messages over ten balanced intents from one seeded split of one corpus, the metrics are exact-match accuracy and macro-F1 (neither a calibration measure), and Banking77 messages are short, English and single-intent. So a gain here says the contract works, not that the adapted model is better on your label set, that it separates intents whose phrases overlap, or that its scores mean anything as probabilities — they remain entailment-derived softmaxes. Fine-tuning towards ten phrases and one template also pulls the model away from general NLI: the adapted scorer still accepts any label set, but its zero-shot behaviour on other labels is changed, and nothing here measures that.

Three things to carry to real data. **Baseline first:** the majority baseline and the frozen model's accuracy and per-label F1 on *your* labels, with *your* phrases and template, are the numbers to read before any adapted one — phrase wording moves zero-shot scores as much as the messages do. **Leakage:** de-duplicate messages across splits (the contract does this case-insensitively) and split by customer or conversation when several messages come from one. **Ceilings:** a message paired with a hypothesis over `MAX_TEXT_TOKENS` is refused at inference and truncated to 256 tokens only during training — long-document classification is out of scope.

Successful execution proves that the recorded repository revision's pipeline modules, carried in this standalone notebook, can acquire and digest-verify the pinned model snapshot, fetch and digest-verify a real labelled corpus, validate the demonstrated dataset contract without leakage, execute the inference contract and a bounded fine-tuning, evaluate against a trivial baseline and the frozen model on an independent split, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, classification quality on any other label set, a calibrated score or acceptance threshold, or production fitness.

**Optional experiments (they do not affect the default path):** set `TRAINABLE_DECODER_LAYERS = 1` and compare the artifact size and the test scores; change `TEMPLATE` (for example `This message is about {}.`) and watch the frozen per-label F1 move before any training; set `MULTI_LABEL = True` in Section 5 and read the independent scores; or bring your own labelled messages through BYOD and read the majority baseline before the adapted number.

## References

- Repository README: https://github.com/kurtvalcorza/bart-mnli-zero-shot-classification-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/bart-mnli-zero-shot-classification-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/bart-mnli-zero-shot-classification-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/facebook/bart-large-mnli
- Upstream code: https://github.com/facebookresearch/fairseq/tree/main/examples/bart
- BART: Denoising Sequence-to-Sequence Pre-training for Natural Language Generation, Translation, and Comprehension (Lewis et al., 2019): https://arxiv.org/abs/1910.13461
- Benchmarking Zero-shot Text Classification: Datasets, Evaluation and Entailment Approach (Yin et al., EMNLP 2019): https://arxiv.org/abs/1909.00161
- Efficient Intent Detection with Dual Sentence Encoders (Casanueva et al., 2020; Banking77, CC BY 4.0): https://arxiv.org/abs/2003.04807
- DIMER Notebook Specification 2.0 and Model Card Specification 1.1 (fleet specs in the ml-worker repository)